<!-- course-header -->
# Lesson 29: Training at scale: GPUs, parallelism and efficient architectures

*Companion notebook for lesson 29. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/29-training-at-scale.html) (it works well on a phone) or [offline](../lessons/29-training-at-scale.html). Worked solutions are in `solutions/29-training-at-scale-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/29-training-at-scale.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

See what really limits training speed and memory on accelerators, how data, tensor and pipeline parallelism and sharding split a model across thousands of GPUs, and which architectural ideas (FlashAttention, mixture of experts, state-space models) make frontier models affordable, all through measurements and simulations you run on a laptop.

**Runs on:** Any computer (CPU). Everything is measured or simulated at small scale; one optional cell uses a GPU if you have one. The notebook runs in a few minutes.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy matplotlib torch

## The idea in one paragraph

When models and data outgrow one device, three things change: speed is limited by moving data, the training state must be split across many GPUs, and architecture choices exist to make both cheaper. Every idea in this notebook is measured or simulated on a laptop CPU.

## How an accelerator spends its time

Accelerators have huge arithmetic throughput fed by a hierarchy of memories: registers, on-chip SRAM, HBM, then other GPUs and other servers. **Arithmetic intensity** is FLOPs performed per byte moved.

### Arithmetic intensity and the roofline

Attainable FLOP/s $= \min(\text{peak}, I \times \text{bandwidth})$. This cell times matrix multiplications of growing size and one large element-wise addition.

In [ ]:
import time
import torch

def bench(fn, repeats=10):
    """Best of several timed runs, in seconds (the best is the least noisy)."""
    fn()                                          # warm-up run
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return min(times)

torch.manual_seed(0)
results = []                                      # (name, FLOP/byte, GFLOP/s, GB/s)
for n in [64, 256, 1024, 2048]:
    a, b, c = torch.randn(n, n), torch.randn(n, n), torch.empty(n, n)
    t = bench(lambda: torch.matmul(a, b, out=c))
    flops, nbytes = 2 * n**3, 3 * n * n * 4       # read a and b, write c (fp32)
    results.append((f"matmul {n}", flops / nbytes, flops / t / 1e9,
                    nbytes / t / 1e9))

x, y, z = torch.randn(2**24), torch.randn(2**24), torch.empty(2**24)
t = bench(lambda: torch.add(x, y, out=z))         # 16.8 million additions
flops, nbytes = x.numel(), 3 * x.numel() * 4      # read x and y, write z
results.append(("add", flops / nbytes, flops / t / 1e9, nbytes / t / 1e9))

print(f"{'operation':<13}{'FLOP/byte':>10}{'GFLOP/s':>10}{'GB/s':>8}")
for name, intensity, gflops, gbs in results:
    print(f"{name:<13}{intensity:10.2f}{gflops:10.1f}{gbs:8.1f}")

On a laptop CPU you will see something like this (your timings will differ):

```text
operation     FLOP/byte   GFLOP/s    GB/s
matmul 64         10.67      30.3     2.8
matmul 256        42.67     123.1     2.9
matmul 1024      170.67     489.8     2.9
matmul 2048      341.33     390.2     1.1
add                0.08       3.6    42.7
```

Plot your measurements against a roofline built from them: the peak is your best matmul, the bandwidth is what the addition achieved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

peak = max(r[2] for r in results)                 # best GFLOP/s you measured
bandwidth = results[-1][3]                        # GB/s achieved by the addition
intensity = np.logspace(-2, 3, 200)
plt.figure(figsize=(6, 4))
roof = np.minimum(peak, intensity * bandwidth)
plt.loglog(intensity, roof, label="your roofline")
for name, i, gflops, _ in results:
    plt.scatter(i, gflops)
    plt.annotate(name, (i, gflops), textcoords="offset points", xytext=(5, -10))
plt.axvline(peak / bandwidth, linestyle=":", color="grey", label="ridge point")
plt.xlabel("arithmetic intensity (FLOPs per byte)")
plt.ylabel("achieved GFLOP/s")
plt.legend()
plt.show()
print(f"ridge point of this machine: about {peak / bandwidth:.0f} FLOPs per byte")

## Number formats and mixed precision

A float spends its bits on a sign, an exponent (range) and a mantissa (precision). PyTorch reports what each format can represent.

In [ ]:
formats = [torch.float32, torch.bfloat16, torch.float16,
           torch.float8_e4m3fn, torch.float8_e5m2]
for dt in formats:
    fi = torch.finfo(dt)
    print(f"{str(dt)[6:]:<14}{fi.bits:>3} bits   max {fi.max:<9.3g}"
          f"smallest normal {fi.tiny:<9.2g}eps {fi.eps:.2g}")

```text
float32        32 bits   max 3.4e+38  smallest normal 1.2e-38  eps 1.2e-07
bfloat16       16 bits   max 3.39e+38 smallest normal 1.2e-38  eps 0.0078
float16        16 bits   max 6.55e+04 smallest normal 6.1e-05  eps 0.00098
float8_e4m3fn   8 bits   max 448      smallest normal 0.016    eps 0.12
float8_e5m2     8 bits   max 5.73e+04 smallest normal 6.1e-05  eps 0.25
```

Low precision fails in two characteristic ways: lost updates (why optimisers keep float32 master weights) and underflow (why fp16 needs loss scaling).

In [ ]:
w32 = torch.tensor(1.0)
w16 = torch.tensor(1.0, dtype=torch.bfloat16)
for _ in range(1000):
    w32 += 1e-4               # a small update, like learning rate x gradient
    w16 += 1e-4
print(w32.item(), w16.item())

grads = torch.tensor([1e-3, 1e-6, 1e-8])
print(grads.half())                         # fp16: the smallest underflows to 0
print(grads.bfloat16())                     # bf16: fine range, coarse precision
scale = 2.0 ** 16                           # loss scaling: scale up first...
print((grads * scale).half().float() / scale)   # ...unscale in fp32: no loss

```text
1.1000165939331055 1.0
tensor([1.0004e-03, 1.0133e-06, 0.0000e+00], dtype=torch.float16)
tensor([9.9945e-04, 9.9838e-07, 1.0012e-08], dtype=torch.bfloat16)
tensor([1.0004e-03, 1.0002e-06, 9.9972e-09])
```

## The memory budget of training

Mixed-precision Adam costs about 16 bytes per parameter (2 bf16 weights, 2 bf16 gradients, 12 for float32 master weights and Adam's two moments), plus activations, estimated here at about 34 bytes per token per unit of width per layer (Korthikanti et al., 2022). Treat the results as lower bounds.

In [ ]:
def training_memory_gb(n_params, n_layers, d_model, seq_len, micro_batch,
                       checkpointing=False):
    """Rough memory to train on ONE device with mixed-precision Adam."""
    states = 16 * n_params   # bf16 weights 2 + bf16 grads 2 + fp32 master, Adam 12
    tokens = seq_len * micro_batch
    if checkpointing:   # keep each layer's bf16 input, rebuild one layer at a time
        acts = (2 * n_layers + 34) * tokens * d_model
    else:               # about 34 bytes per token per unit of width, in every layer
        acts = 34 * n_layers * tokens * d_model
    return states / 1e9, acts / 1e9

models = {"1B": (1e9, 16, 2048), "7B": (7e9, 32, 4096), "70B": (70e9, 80, 8192)}
print(f"{'model':<6}{'states GB':>10}{'activations GB':>16}{'checkpointed GB':>17}")
for name, (n, layers, d) in models.items():
    states, acts = training_memory_gb(n, layers, d, seq_len=4096, micro_batch=1)
    _, acts_ckpt = training_memory_gb(n, layers, d, 4096, 1, checkpointing=True)
    print(f"{name:<6}{states:10.0f}{acts:16.1f}{acts_ckpt:17.1f}")

```text
model  states GB  activations GB  checkpointed GB
1B            16             4.6              0.6
7B           112            18.3              1.6
70B         1120            91.3              6.5
```

### Activation checkpointing

Keep only each layer's input in the forward pass and recompute the insides during backward. The code counts the bytes autograd saves for the backward pass and compares gradients.

In [ ]:
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

class MLPStack(nn.Module):
    def __init__(self, d=256, n_layers=8, use_checkpoint=False):
        super().__init__()
        self.layers = nn.ModuleList(
            nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
            for _ in range(n_layers))
        self.use_checkpoint = use_checkpoint

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint:   # drop the insides, recompute in backward
                x = x + checkpoint(layer, x, use_reentrant=False)
            else:
                x = x + layer(x)
        return x

def backward_and_measure(model, x):
    """Run forward and backward; return gradients and MB saved for backward."""
    saved = 0
    def pack(t):
        nonlocal saved
        if not isinstance(t, nn.Parameter):       # count activations, not weights
            saved += t.numel() * t.element_size()
        return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        loss = model(x).pow(2).mean()
    loss.backward()
    return [p.grad for p in model.parameters()], saved / 1e6

torch.manual_seed(0)
x = torch.randn(512, 256)
plain = MLPStack()
ckpt = MLPStack(use_checkpoint=True)
ckpt.load_state_dict(plain.state_dict())          # identical weights
g1, mb1 = backward_and_measure(plain, x)
g2, mb2 = backward_and_measure(ckpt, x)
print(f"saved for backward: {mb1:.1f} MB plain, {mb2:.1f} MB checkpointed")
print("identical gradients:", all(torch.equal(a, b) for a, b in zip(g1, g2)))

```text
saved for backward: 54.0 MB plain, 4.7 MB checkpointed
identical gradients: True
```

Optional: on a GPU you can measure peak memory directly. Nothing later depends on this cell.

In [ ]:
# Needs a GPU runtime
# The same comparison on a GPU, measured directly as peak memory allocated.
device = "cuda"
xg = torch.randn(8192, 1024, device=device)
for use in (False, True):
    torch.manual_seed(0)
    m = MLPStack(d=1024, n_layers=16, use_checkpoint=use).to(device)
    torch.cuda.reset_peak_memory_stats()
    m(xg).pow(2).mean().backward()
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"checkpointing={use}: peak {peak_mb:,.0f} MB")

### Gradient accumulation

Run several micro-batches, let gradients add up in `.grad`, step once. Divide each loss by the number of micro-batches.

In [ ]:
import copy
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
X, y = torch.randn(256, 20), torch.randn(256, 1)
loss_fn = nn.MSELoss()                            # a mean over the batch

model.zero_grad()
loss_fn(model(X), y).backward()                   # one batch of 256
full = [p.grad.clone() for p in model.parameters()]

model.zero_grad()
steps = 4
for Xm, ym in zip(X.chunk(steps), y.chunk(steps)):    # four micro-batches of 64
    (loss_fn(model(Xm), ym) / steps).backward()       # .grad keeps adding up
accum = [p.grad.clone() for p in model.parameters()]
print("max difference:",
      max((a - b).abs().max().item() for a, b in zip(full, accum)))

```text
max difference: 1.043081283569336e-07
```

### Hands-on: when does a linear layer become compute-bound?

1. Write `linear_intensity(b, d)` for $y = xW$ with $x$ of shape $(b, d)$ and $W$ of shape $(d, d)$, in bf16 (2 bytes per number). Count $2bd^2$ FLOPs, and bytes for reading $x$ and $W$ and writing $y$.
2. For $d = 8192$, print the intensity for $b = 1, 16, 128, 1024, 8192$. What is it roughly equal to while $b$ is much smaller than $d$?
3. Find the smallest $b$ at which the layer passes the ridge point of the illustrative accelerator (1,000 TFLOP/s, 3 TB/s).
4. Measure it: with $d = 2048$ in float32, time `x @ W` with `bench` for $b$ from 1 to 4,096, print the GFLOP/s and plot them.

You are done when you can explain why generating text for one user at a time is memory-bound while training is compute-bound.

In [ ]:
# Your code here

## Data parallelism

Every worker holds a full replica, processes a shard of the batch, and the gradients are averaged with an all-reduce. Simulate four workers with the model and data from the previous cell.

In [ ]:
n_workers = 4
workers = [copy.deepcopy(model) for _ in range(n_workers)]   # a full replica each
for w, Xs, ys in zip(workers, X.chunk(n_workers), y.chunk(n_workers)):
    w.zero_grad()
    loss_fn(w(Xs), ys).backward()    # each worker sees only its shard of the batch

# all-reduce: every worker ends up with the average of all workers' gradients
for params in zip(*(w.parameters() for w in workers)):
    avg = torch.stack([p.grad for p in params]).mean(dim=0)
    for p in params:
        p.grad = avg.clone()

dp = [p.grad for p in workers[0].parameters()]
print("max difference from full batch:",
      max((a - b).abs().max().item() for a, b in zip(full, dp)))

```text
max difference from full batch: 1.043081283569336e-07
```

### How an all-reduce works

A ring all-reduce is a reduce-scatter followed by an all-gather. Count how much each worker sends.

In [ ]:
import numpy as np

def ring_all_reduce(arrays):
    """Sum equal-length arrays held by n workers by passing chunks round a ring."""
    n = len(arrays)
    chunks = [np.array_split(a.astype(float), n) for a in arrays]
    sent = 0
    # 1. reduce-scatter: after n-1 steps, worker i owns the total of chunk (i+1) % n
    for step in range(n - 1):
        msgs = [(i, (i - step) % n) for i in range(n)]
        data = [chunks[i][c].copy() for i, c in msgs]   # everyone sends at once
        for (i, c), d in zip(msgs, data):
            chunks[(i + 1) % n][c] += d                 # the neighbour adds it in
            sent += d.size
    # 2. all-gather: pass each finished chunk round the ring
    for step in range(n - 1):
        msgs = [(i, (i + 1 - step) % n) for i in range(n)]
        data = [chunks[i][c].copy() for i, c in msgs]
        for (i, c), d in zip(msgs, data):
            chunks[(i + 1) % n][c] = d                  # the neighbour copies it
            sent += d.size
    return [np.concatenate(c) for c in chunks], sent / n    # values sent per worker

rng = np.random.default_rng(0)
size = 120_000                                    # gradient values per worker
for n in [2, 4, 8, 64]:
    grads = [rng.normal(size=size) for _ in range(n)]
    total = np.sum(grads, axis=0)
    result, per_worker = ring_all_reduce(grads)
    ok = all(np.allclose(r, total) for r in result)
    print(f"{n:>3} workers: correct={ok}, "
          f"each sends {per_worker / size:.2f} x its gradient")

```text
  2 workers: correct=True, each sends 1.00 x its gradient
  4 workers: correct=True, each sends 1.50 x its gradient
  8 workers: correct=True, each sends 1.75 x its gradient
 64 workers: correct=True, each sends 1.97 x its gradient
```

## ZeRO and FSDP: stop storing everything N times

Stage 1 shards the optimiser state, stage 2 also the gradients, stage 3 (FSDP) also the weights.

In [ ]:
def zero_memory_gb(n_params, n_gpus, stage):
    """Training-state memory per GPU (bf16 weights and grads, fp32 Adam)."""
    weights, grads, optim = 2 * n_params, 2 * n_params, 12 * n_params
    if stage >= 1: optim /= n_gpus       # ZeRO-1: shard the optimiser state
    if stage >= 2: grads /= n_gpus       # ZeRO-2: ...and the gradients
    if stage >= 3: weights /= n_gpus     # ZeRO-3 (FSDP): ...and the weights too
    return (weights + grads + optim) / 1e9

print(f"{'model':<6}{'GPUs':>5}"
      + "".join(f"{'stage ' + str(s):>10}" for s in range(4)))
for name, n in [("7B", 7e9), ("70B", 70e9)]:
    for gpus in [8, 64]:
        row = "".join(f"{zero_memory_gb(n, gpus, s):10.1f}" for s in range(4))
        print(f"{name:<6}{gpus:>5}{row}")

```text
model  GPUs   stage 0   stage 1   stage 2   stage 3
7B        8     112.0      38.5      26.2      14.0
7B       64     112.0      29.3      15.5       1.8
70B       8    1120.0     385.0     262.5     140.0
70B      64    1120.0     293.1     155.3      17.5
```

## Model parallelism: splitting the model itself

### Tensor parallelism

Split the MLP's first matrix by columns and the second by rows: each GPU works independently and one all-reduce sums the partial results.

In [ ]:
import torch.nn.functional as F
torch.manual_seed(0)
tokens, d, d_ff, n_gpus = 6, 16, 64, 4
X = torch.randn(tokens, d)
A, B = torch.randn(d, d_ff), torch.randn(d_ff, d)    # the MLP's two weight matrices
full = F.gelu(X @ A) @ B                             # one device does it all

A_parts = A.chunk(n_gpus, dim=1)    # column split: each GPU gets d_ff/4 hidden units
B_parts = B.chunk(n_gpus, dim=0)     # row split: the matching rows of B
partial = [F.gelu(X @ a) @ b for a, b in zip(A_parts, B_parts)]  # no talking needed
out = torch.stack(partial).sum(dim=0)                # one all-reduce at the end
print("column-then-row split matches:", torch.allclose(out, full, atol=1e-4))

# Splitting A by rows instead gives partial sums of X @ A, and GELU of a partial
# sum is not part of the answer: you would need an all-reduce before the GELU.
X_parts, A_rows = X.chunk(n_gpus, dim=1), A.chunk(n_gpus, dim=0)
wrong = sum(F.gelu(xp @ ar) for xp, ar in zip(X_parts, A_rows))
print("GELU of partial sums matches:",
      torch.allclose(wrong, F.gelu(X @ A), atol=1e-4))

```text
column-then-row split matches: True
GELU of partial sums matches: False
```

### Pipeline parallelism

With $p$ stages and $m$ micro-batches, the idle fraction (the bubble) is $(p-1)/(m+p-1)$.

In [ ]:
def bubble_fraction(stages, micro_batches):
    return (stages - 1) / (micro_batches + stages - 1)

for m in [1, 4, 16, 64]:
    print(f"4 stages, {m:>2} micro-batches: {bubble_fraction(4, m):.0%} idle")

def gpipe_schedule(stages, micro_batches):
    """Print who works when: F = forward, B = backward, . = idle."""
    slots = 2 * (micro_batches + stages - 1)
    for s in range(stages):
        row = ["."] * slots
        for j in range(micro_batches):
            row[s + j] = f"F{j}"                        # forwards flow down
            back = micro_batches + stages - 1 + (stages - 1 - s) + j
            row[back] = f"B{j}"                         # backwards flow up
        print(f"GPU {s}  " + " ".join(f"{c:<2}" for c in row).rstrip())

gpipe_schedule(4, 4)

```text
4 stages,  1 micro-batches: 75% idle
4 stages,  4 micro-batches: 43% idle
4 stages, 16 micro-batches: 16% idle
4 stages, 64 micro-batches: 4% idle
GPU 0  F0 F1 F2 F3 .  .  .  .  .  .  B0 B1 B2 B3
GPU 1  .  F0 F1 F2 F3 .  .  .  .  B0 B1 B2 B3 .
GPU 2  .  .  F0 F1 F2 F3 .  .  B0 B1 B2 B3 .  .
GPU 3  .  .  .  F0 F1 F2 F3 B0 B1 B2 B3 .  .  .
```

How the bubble shrinks as you add micro-batches, for different pipeline depths:

In [ ]:
plt.figure(figsize=(6, 3.5))
mb = np.arange(1, 129)
for p in [2, 4, 8, 16]:
    plt.plot(mb, [bubble_fraction(p, m) for m in mb], label=f"{p} stages")
plt.xscale("log")
plt.xlabel("micro-batches per step")
plt.ylabel("fraction of time idle")
plt.legend()
plt.show()

### Sequence and context parallelism

Sequence parallelism splits layer norm and dropout along the sequence inside tensor-parallel groups; context parallelism splits attention over a long sequence, passing key and value blocks round a ring and combining them with the online softmax you will implement in session 3.

## Putting it together: 3-D parallelism

Tensor parallelism inside a server (fast NVLink), pipeline stages across servers, data parallelism with sharding across everything else. Put the chattiest parallelism on the fastest wires.

## Compute budgets and MFU

Training time $\approx 6ND / (\text{GPUs} \times \text{peak} \times \text{MFU})$. The calculator assumes an illustrative 1 petaFLOP/s per GPU and 40% MFU.

In [ ]:
def training_time(n_params, n_tokens, n_gpus, peak_flops=1e15, mfu=0.4):
    """FLOPs, days and GPU-hours for a dense transformer, from C = 6 N D."""
    flops = 6 * n_params * n_tokens
    seconds = flops / (n_gpus * peak_flops * mfu)
    return flops, seconds / 86400, n_gpus * seconds / 3600

runs = [("1B on 20B tokens", 1e9, 20e9, 8),
        ("7B on 2T tokens", 7e9, 2e12, 512),
        ("70B on 15T tokens", 70e9, 15e12, 8192)]
for name, n, d_tok, gpus in runs:
    flops, days, gpu_hours = training_time(n, d_tok, gpus)
    print(f"{name:<18} {flops:.1e} FLOPs {gpus:>5} GPUs {days:6.1f} days"
          f" {gpu_hours:11,.0f} GPU-hours")

```text
1B on 20B tokens   1.2e+20 FLOPs     8 GPUs    0.4 days          83 GPU-hours
7B on 2T tokens    8.4e+22 FLOPs   512 GPUs    4.7 days      58,333 GPU-hours
70B on 15T tokens  6.3e+24 FLOPs  8192 GPUs   22.3 days   4,375,000 GPU-hours
```

## What goes wrong at scale

Loss spikes, hardware failures, terabyte checkpoints, stragglers and silent errors, and non-determinism from reduction order. See the lesson for the defences.

### Hands-on: plan a 13B training run

You have 64 GPUs with 80 GB each, and a dense 13B model with 40 layers of width 5,120, to train on 1 trillion tokens in sequences of 4,096 with one sequence per micro-batch. Use the functions from this lesson.

1. For ZeRO stages 0 to 3, print the training state per GPU and how much of the 80 GB is left.
2. Compute the activation memory with and without checkpointing. What is the lowest ZeRO stage that fits without checkpointing?
3. How many days does training take at 40% MFU on 64 GPUs, and on 512?
4. With 8 pipeline stages, how many micro-batches keep the bubble under 10%?
5. Stretch: measure your laptop's MFU. Time a few AdamW steps of `MLPStack` on a batch of 512, compute $6 \times \text{parameters} \times \text{tokens per second}$, and divide by the matmul peak you measured.

In [ ]:
# Your code here

## FlashAttention: attention that respects the memory hierarchy

Compute exact attention one tile at a time, with a running maximum $m$, running sum $\ell$ and running output $o$ per row (the online softmax), and never hold the full $T \times T$ matrix.

In [ ]:
def naive_attention(q, k, v):
    scores = q @ k.T / q.shape[-1] ** 0.5        # (T, T): the whole matrix exists
    return scores.softmax(dim=-1) @ v

def tiled_attention(q, k, v, block=128):
    """Exact attention, holding one (block x block) tile of scores at a time."""
    T, d = q.shape
    out = torch.empty_like(q)
    for i in range(0, T, block):                 # a block of queries stays on chip
        qi = q[i:i + block] / d ** 0.5
        m = torch.full((len(qi), 1), float("-inf"))   # running max of each row
        l = torch.zeros(len(qi), 1)                   # running sum of exp(s - m)
        o = torch.zeros(len(qi), d)                   # running weighted sum of v
        for j in range(0, T, block):             # stream key/value blocks past it
            s = qi @ k[j:j + block].T            # the only scores we ever hold
            m_new = torch.maximum(m, s.max(dim=-1, keepdim=True).values)
            rescale = torch.exp(m - m_new)       # correct what was summed so far
            p = torch.exp(s - m_new)
            l = l * rescale + p.sum(dim=-1, keepdim=True)
            o = o * rescale + p @ v[j:j + block]
            m = m_new
        out[i:i + block] = o / l                 # normalise once, at the end
    return out

torch.manual_seed(0)
T, d = 2048, 64
q, k, v = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)
ref = naive_attention(q, k, v)
tiled = tiled_attention(q, k, v)
print("tiled matches naive:", torch.allclose(tiled, ref, atol=1e-5))
fused = F.scaled_dot_product_attention(q[None], k[None], v[None])[0]
print("PyTorch's fused kernel matches:", torch.allclose(fused, ref, atol=1e-5))
print(f"largest score matrix: naive {T * T:,} values, tiled {128 * 128:,}")

```text
tiled matches naive: True
PyTorch's fused kernel matches: True
largest score matrix: naive 4,194,304 values, tiled 16,384
```

## Mixture of experts

A router sends each token to its top-$k$ experts. Count total and active parameters, and watch the load-balancing loss react when the routing is skewed.

In [ ]:
class MoE(nn.Module):
    """A mixture-of-experts MLP: each token goes to its top-k experts."""
    def __init__(self, d=64, d_ff=256, n_experts=8, k=2):
        super().__init__()
        self.k, self.n_experts = k, n_experts
        self.router = nn.Linear(d, n_experts, bias=False)
        self.experts = nn.ModuleList(
            nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d))
            for _ in range(n_experts))

    def forward(self, x):                                   # x: (tokens, d)
        probs = self.router(x).softmax(dim=-1)              # (tokens, n_experts)
        weight, chosen = probs.topk(self.k, dim=-1)         # k experts per token
        weight = weight / weight.sum(dim=-1, keepdim=True)
        out = torch.zeros_like(x)
        for e, expert in enumerate(self.experts):
            token, slot = (chosen == e).nonzero(as_tuple=True)   # who picked e?
            if len(token):
                y_e = weight[token, slot, None] * expert(x[token])
                out.index_add_(0, token, y_e)
        # load-balancing loss (Switch Transformer): the fraction of tokens sent to
        # each expert times its mean router probability; 1.0 when perfectly even
        self.load = torch.bincount(chosen.flatten(), minlength=self.n_experts)
        frac = self.load / chosen.numel()
        self.aux_loss = self.n_experts * (frac * probs.mean(dim=0)).sum()
        return out

torch.manual_seed(0)
moe = MoE()
tokens = torch.randn(1000, 64)
y_moe = moe(tokens)
per_expert = sum(p.numel() for p in moe.experts[0].parameters())
total = sum(p.numel() for p in moe.parameters())
active = moe.router.weight.numel() + moe.k * per_expert
print(f"total {total:,} parameters, active per token {active:,} "
      f"({active / total:.0%})")
print("tokens per expert:", moe.load.tolist(), f"loss {moe.aux_loss.item():.3f}")

w0 = moe.router.weight[0].detach()
moe(tokens + 3 * w0 / w0.norm())                 # push every token towards expert 0
print("after skewing:    ", moe.load.tolist(), f"loss {moe.aux_loss.item():.3f}")

```text
total 265,216 parameters, active per token 66,688 (25%)
tokens per expert: [248, 251, 240, 235, 228, 281, 252, 265] loss 1.001
after skewing:     [978, 43, 41, 316, 80, 72, 73, 397] loss 1.961
```

## Long context and state-space models

Per layer, linear layers cost about $24Td^2$ forward FLOPs and attention about $4T^2d$: attention dominates once $T > 6d$. RoPE scaling (position interpolation, YaRN) lets models extend their context after pretraining. A selective scan instead carries a fixed-size state with a linear recurrence. Compare how its time grows with $T$ against naive attention.

In [ ]:
def selective_scan(x, delta, A, B, C):
    """h_t = exp(delta_t A) h_(t-1) + delta_t B_t x_t, then y_t = h_t C_t."""
    decay = torch.exp(delta[:, :, None] * A)              # (T, d, n), in (0, 1)
    h = delta[:, :, None] * B[:, None, :] * x[:, :, None]  # (T, d, n) inputs
    for t in range(1, len(x)):                             # the sequential part
        h[t] += decay[t] * h[t - 1]
    return torch.einsum("tdn,tn->td", h, C)                # read each state out

torch.manual_seed(0)
d, n = 64, 16                          # 64 channels, a 16-number state for each
A = -torch.rand(d, n) - 0.5                      # negative, so the state decays
W_delta, W_B, W_C = (torch.randn(d, k_out) / 8 for k_out in (d, n, n))

def ssm_layer(x):          # "selective": decay, input and readout all depend on x
    return selective_scan(x, F.softplus(x @ W_delta), A, x @ W_B, x @ W_C)

timings = []
print(f"{'T':>6}{'attention ms':>14}{'scan ms':>10}{'scores MB':>11}")
for T in [1024, 2048, 4096, 8192]:
    x = torch.randn(T, d)
    t_attn = bench(lambda: naive_attention(x, x, x), repeats=3) * 1000
    t_scan = bench(lambda: ssm_layer(x), repeats=3) * 1000
    timings.append((T, t_attn, t_scan))
    print(f"{T:>6}{t_attn:14.1f}{t_scan:10.1f}{T * T * 4 / 1e6:11.0f}")

On a laptop CPU you will see something like this (your timings will differ):

```text
     T  attention ms   scan ms  scores MB
  1024           1.3      19.3          4
  2048           5.0      44.9         17
  4096          24.5      66.4         67
  8192         135.0     157.7        268
```

In [ ]:
Ts, t_attn, t_scan = zip(*timings)
plt.figure(figsize=(6, 3.5))
plt.loglog(Ts, t_attn, "o-", label="naive attention")
plt.loglog(Ts, t_scan, "o-", label="selective scan")
plt.xlabel("sequence length T")
plt.ylabel("time (ms)")
plt.legend()
plt.show()

## Smaller caches and lower precision

GQA shares key and value heads; MLA caches a compressed latent per token. FP8 and block-scaled formats carry more of the compute, with higher precision kept for master weights and optimiser state.

## Reading a model card's architecture section

Memory follows total parameters, compute follows active parameters, the KV cache follows layers, KV heads and head size, and the training budget follows $6ND$.

### Hands-on: read a model card

A made-up model card lists 32 layers of width 4,096; a vocabulary of 128,000 with separate input and output embeddings; 32 query heads and 8 KV heads of dimension 128; and in every layer 64 SwiGLU experts of hidden size 1,536, with 4 active per token. A SwiGLU expert has three matrices of shape $d \times d_{\text{expert}}$.

1. Write `count_params(config)` that returns total and active parameters (ignore norms and biases).
2. How much memory do the weights need in bf16?
3. What is the KV cache per token in bf16, and what would it be with 32 KV heads?
4. How many FLOPs would training on 10 trillion tokens take, and how does that compare with a dense model of the same total size?

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/29-training-at-scale.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.